The code will have many drawbacks
1. sender, receiver, subject ignored
2. body text not formatted properly, should be tokenized and lemmatized
3. from corpus, remove features which are too rare
4. train test set splitted randomly and not stratified
5. folds not created for cross validation
6. no transformer created
7. assuming all structures are dfs
8. ensemble method not created. poor model

In [78]:
import numpy as np
import pandas as pd
from conf.user_info import USER_EMAIL_ID

In [79]:
df = pd.read_csv('../data/training_data.csv', sep='~', index_col=0)
condition = df['sender'] == f'{USER_EMAIL_ID}'
df.drop(df[condition].index, inplace=True)

In [80]:
df_reindexed = df.reset_index(drop=True)
df_reindexed.index

RangeIndex(start=0, stop=969, step=1)

In [81]:
from sklearn.preprocessing import MultiLabelBinarizer
import json

with open('../data/label_dict.txt', 'r') as file:
    label_str = file.read()

all_labels = json.loads(label_str.replace('\'', '\"'))
all_labels

{'CHAT': 'CHAT',
 'SENT': 'SENT',
 'INBOX': 'INBOX',
 'IMPORTANT': 'IMPORTANT',
 'TRASH': 'TRASH',
 'DRAFT': 'DRAFT',
 'SPAM': 'SPAM',
 'CATEGORY_FORUMS': 'CATEGORY_FORUMS',
 'CATEGORY_UPDATES': 'CATEGORY_UPDATES',
 'CATEGORY_PERSONAL': 'CATEGORY_PERSONAL',
 'CATEGORY_PROMOTIONS': 'CATEGORY_PROMOTIONS',
 'CATEGORY_SOCIAL': 'CATEGORY_SOCIAL',
 'STARRED': 'STARRED',
 'UNREAD': 'UNREAD',
 'Label_1121764005411591504': 'Class materials',
 'Label_1381130677208788327': 'academics',
 'Label_218404524901403708': 'MTE',
 'Label_2692894171255220949': 'Techincal Clubs',
 'Label_3281058460844688860': 'jobs and internship',
 'Label_3628272813817210666': 'quarantined',
 'Label_4319519781275842575': 'treks',
 'Label_464282237945132064': 'AAO',
 'Label_4724788481168709854': 'ETE',
 'Label_4818832377804158016': 'institute updates',
 'Label_5079845490397963276': 'cultural',
 'Label_5249056522118261244': 'SCSP',
 'Label_5554613578351500543': 'commercial',
 'Label_5975700729676723285': 'bhawan notifs',
 'L

In [82]:
import re

label_list = [key for key in all_labels.keys() if re.match('Label_[0-9]', key)]
label_list

['Label_1121764005411591504',
 'Label_1381130677208788327',
 'Label_218404524901403708',
 'Label_2692894171255220949',
 'Label_3281058460844688860',
 'Label_3628272813817210666',
 'Label_4319519781275842575',
 'Label_464282237945132064',
 'Label_4724788481168709854',
 'Label_4818832377804158016',
 'Label_5079845490397963276',
 'Label_5249056522118261244',
 'Label_5554613578351500543',
 'Label_5975700729676723285',
 'Label_632409477860964109',
 'Label_7245492495222995092',
 'Label_8087426001935712506',
 'Label_8099758025489700714',
 'Label_8173601450858728231',
 'Label_880764872117155350',
 'Label_9086749827465587050']

In [83]:
mlb = MultiLabelBinarizer(classes=label_list)
labels_array = [list(st.split(',')) for st in df['labels']]
labels = mlb.fit_transform(labels_array)
labels

C:\Users\msing\.conda\envs\gmail_organizer\lib\site-packages\sklearn\preprocessing\_label.py:895: UserWarning: unknown class(es) ['CATEGORY_FORUMS', 'CATEGORY_PERSONAL', 'CATEGORY_PROMOTIONS', 'CATEGORY_UPDATES', 'IMPORTANT', 'INBOX', 'STARRED', 'UNREAD'] will be ignored
  warnings.warn(


array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [84]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='constant', fill_value='')
df_reindexed_imputed = pd.DataFrame(imputer.fit_transform(df_reindexed))

In [85]:
df_reindexed_imputed

,0,1,2,3,4,5
0,18692a9543f56391,gensec.cult@iitr.ac.in,students-notices@iitr.ac.in,Raabta | Short Film | Cinesec IITR,Greetings Everyone! Cinesec IITR is here again...,"Label_5079845490397963276,CATEGORY_FORUMS,INBOX"
1,186929e4cd7d6beb,info@send.grammarly.com,aakash_ks@mfs.iitr.ac.in,[BULK] Quick tip 3: Write to engage your reader,~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~...,"CATEGORY_PROMOTIONS,Label_5554613578351500543,..."
2,186915990080ce80,noreply-emailsupport@iitr.ac.in,aakash_ks@mfs.iitr.ac.in,Updates: Your List of Quarantined Emails since...,Spam Quarantine Notification Spam Quarantine ...,"Label_3628272813817210666,IMPORTANT,CATEGORY_U..."
3,186914148d87644e,head@ph.iitr.ac.in,"faculty-notices@iitr.ac.in,students-notices@ii...",PSIM Conference India 2023: A warm invitation,"Dear Colleagues and Students,Warm greetings f...","IMPORTANT,CATEGORY_PERSONAL,INBOX"
4,18690deee583476b,greenoffice@iitr.ac.in,"staff-notices@iitr.ac.in,students-notices@iitr...",solar energy in the campus,Apology for typo error in title and solar PV c...,"UNREAD,IMPORTANT,CATEGORY_PERSONAL,INBOX"
...,...,...,...,...,...,...
964,18443096fea03e50,gensec.cult@iitr.ac.in,students-notices@iitr.ac.in,Choreography and Dance Section Recruitments | ...,Hola IIT-R freshers! Do you guys love dancing ...,"Label_5079845490397963276,Label_81736014508587..."
965,18442dd147d26785,gensec.hostel@iitr.ac.in,students-notices@iitr.ac.in,Stall setup for formal clothing near MAC,Greetings from Hostel Council!! To make an exe...,"IMPORTANT,CATEGORY_PERSONAL,INBOX"
966,18442d61df86846c,noreply-emailsupport@iitr.ac.in,aakash_ks@mfs.iitr.ac.in,Updates: Your List of Quarantined Emails since...,Spam Quarantine Notification Spam Quarantine ...,"Label_3628272813817210666,IMPORTANT,CATEGORY_U..."
967,18442841dd0a2dbe,head@cy.iitr.ac.in,"faculty-notices@iitr.ac.in,students-notices@ii...",Invitation to Inauguration of ChemDay - Nov. 0...,"Dear Colleagues and Students, The Department o...","IMPORTANT,CATEGORY_PERSONAL,INBOX"


In [86]:
# import neattext as nt
# import neattext.functions as nfx
#
# corpus = df_reindexed_imputed[4].apply(nfx.remove_stopwords)

In [87]:
corpus = df_reindexed_imputed[4]
corpus

0      Greetings Everyone! Cinesec IITR is here again...
1      ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~...
2       Spam Quarantine Notification Spam Quarantine ...
3       Dear Colleagues and Students,Warm greetings f...
4      Apology for typo error in title and solar PV c...
                             ...                        
964    Hola IIT-R freshers! Do you guys love dancing ...
965    Greetings from Hostel Council!! To make an exe...
966     Spam Quarantine Notification Spam Quarantine ...
967    Dear Colleagues and Students, The Department o...
968    Dear students, Please find the attached notice...
Name: 4, Length: 969, dtype: object

In [94]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import RegexpTokenizer
from nltk.stem import WordNetLemmatizer

tfidf = TfidfVectorizer(
    max_df=0.8,
    min_df=0.05
)
X_features_body = tfidf.fit_transform(corpus).toarray()
tfidf.inverse_transform(X_features_body)

[array(['and', 'be', 'by', 'com', 'everyone', 'follow', 'for', 'greetings',
        'here', 'https', 'iit', 'iitr', 'instagram', 'is', 'link', 'not',
        'on', 'online', 'only', 'our', 'regards', 'section', 'this',
        'time', 'us', 'was', 'with'], dtype='<U13'),
 array(['2023', 'about', 'and', 'can', 'click', 'com', 'email', 'for',
        'from', 'have', 'https', 'it', 'more', 'on', 'one', 'or', 'please',
        'received', 'so', 'such', 'that', 'these', 'this', 'time', 'view',
        'want', 'we', 'web', 'which', 'with', 'you', 'your'], dtype='<U13'),
 array(['10', '2023', '30', 'above', 'ac', 'access', 'after', 'all', 'and',
        'any', 'are', 'as', 'be', 'been', 'below', 'by', 'click',
        'contact', 'copy', 'date', 'day', 'dear', 'do', 'email', 'feb',
        'following', 'from', 'has', 'have', 'https', 'if', 'iitr', 'into',
        'last', 'links', 'message', 'need', 'new', 'not', 'note', 'on',
        'only', 'open', 'please', 'portal', 'received', 'see', 'sent

In [95]:
X = pd.DataFrame(X_features_body)
y = pd.DataFrame(labels)

print(X.shape,'\n', y.shape)

(969, 357) 
 (969, 21)


In [96]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lst = [X_train, X_test, y_train, y_test]
for i in lst:
    print(i.shape, '\n')


(775, 357) 

(194, 357) 

(775, 21) 

(194, 21) 



In [104]:
from skmultilearn.problem_transform import BinaryRelevance
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [119]:
binary_rel_clf = BinaryRelevance(classifier=SVC())
binary_rel_clf.fit(X_train, y_train)
br_predictions = binary_rel_clf.predict(X_test)

In [120]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test, br_predictions)

0.7268041237113402

In [121]:
# to check how much the model is overfitting
test_prediction = binary_rel_clf.predict(X_train)
accuracy_score(y_train, test_prediction)

0.8348387096774194

In [118]:
knn_clf = KNeighborsClassifier()
knn_clf.fit(X_train, y_train)
knn_predictions = knn_clf.predict(X_test)
accuracy_score(y_test, knn_predictions)

0.7216494845360825

In [122]:
knn_train_pr = knn_clf.predict(X_train)
accuracy_score(y_train, knn_train_pr)

0.7148387096774194